# BudgetMem Revision Experiments

Three experiments needed for paper revision:
1. **Per-feature ablation** on medium docs (leave-one-out, ~2 hours)
2. **LLMLingua-2 on medium docs** (~1 hour)
3. **Qasper benchmark** on real academic papers (~2-3 hours)

Run all cells top to bottom. Results saved to Google Drive.

## Part 0: Setup

In [ ]:
!pip install -q transformers accelerate datasets rank-bm25 nltk scikit-learn tqdm llmlingua

import torch
import numpy as np
import json
import re
import string
import time
import random
import os
from datetime import datetime
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi
from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Save locally (no Google Drive needed)
PROJECT_DIR = "/content/budgetmem_results"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

# Optional: mount Drive if you want persistent storage
# Uncomment the next 3 lines if Drive mount works for you:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = "/content/drive/MyDrive/BudgetMem_Revision"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Results will be saved to {PROJECT_DIR}/results/")
print("To download results: click the folder icon in Colab sidebar -> navigate to /content/budgetmem_results/results/")


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Load the model once, reuse everywhere
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Model loaded.")

## Shared utilities

In [ ]:
# ── F1 scoring (SQuAD-style, same as original experiments) ──

def normalize_answer(s):
    """Lowercase, strip articles, punctuation, extra whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s

def compute_f1(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    truth_tokens = normalize_answer(ground_truth).split()
    if not pred_tokens or not truth_tokens:
        return float(pred_tokens == truth_tokens)
    common = set(pred_tokens) & set(truth_tokens)
    if not common:
        return 0.0
    prec = len(common) / len(pred_tokens)
    rec  = len(common) / len(truth_tokens)
    return 2 * prec * rec / (prec + rec)


# ── Answer generation ──

def generate_answer(context, question, max_ctx_chars=8000):
    """Generate an answer from context + question using the loaded Llama model."""
    prompt = f"""Context: {context[:max_ctx_chars]}

Question: {question}

Answer the question based only on the context above. Be concise.

Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,            # greedy, matching paper
            pad_token_id=tokenizer.eos_token_id
        )
    answer = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return answer.strip()


# ── Chunking ──

def chunk_document(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk.split()) > 10:
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


# ── BM25 retrieval ──

def retrieve_bm25(query, chunks, top_k=3):
    if not chunks:
        return []
    tokenized = [c.split() for c in chunks]
    bm25 = BM25Okapi(tokenized)
    scores = bm25.get_scores(query.split())
    k = min(top_k, len(chunks))
    top_idx = np.argsort(scores)[-k:][::-1]
    return [chunks[i] for i in top_idx]


print("Utilities loaded.")

In [ ]:
# ── BudgetMem salience scorer (matches paper exactly: 6 features) ──

DISCOURSE_MARKERS = [
    'however', 'therefore', 'moreover', 'furthermore', 'consequently',
    'nevertheless', 'additionally', 'specifically', 'importantly',
    'in conclusion', 'on the other hand', 'as a result', 'for example',
    'in contrast', 'meanwhile', 'subsequently', 'nonetheless',
    'accordingly', 'hence', 'thus', 'indeed', 'notably',
    'in particular', 'conversely', 'alternatively', 'likewise',
    'similarly', 'in summary', 'to summarize', 'overall',
    'in other words', 'that is', 'namely', 'first', 'second',
    'third', 'finally', 'next', 'then', 'afterward',
    'before', 'after', 'during', 'while', 'although',
    'despite', 'regardless', 'provided that'
]

# Default weights from the paper
DEFAULT_WEIGHTS = {
    'entity_density':   0.20,
    'tfidf_importance': 0.20,
    'position_bias':    0.15,
    'numerical_density': 0.15,
    'discourse_markers': 0.10,
    'question_presence': 0.10,
}


def compute_features(chunks):
    """
    Compute all 6 features for a list of chunks.
    Returns a dict of arrays, each of length len(chunks), normalized to [0,1].
    """
    n = len(chunks)
    if n == 0:
        return {}

    # 1. Entity density: capitalized-word ratio (spaCy proxy)
    entity_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        words = c.split()
        if words:
            entity_raw[i] = sum(1 for w in words if w and w[0].isupper()) / len(words)

    # 2. TF-IDF importance: mean TF-IDF per chunk
    tfidf_raw = np.zeros(n)
    try:
        vec = TfidfVectorizer(max_features=500, stop_words='english')
        mat = vec.fit_transform(chunks)
        tfidf_raw = np.array(mat.mean(axis=1)).flatten()
    except Exception:
        pass

    # 3. Position bias: U-shaped (favors beginning and end)
    position_raw = np.zeros(n)
    for i in range(n):
        position_raw[i] = min(1.0, max(0.0, 1.3 - 2.0 * abs(i / n - 0.5)))

    # 4. Numerical density: ratio of numeric tokens
    number_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        words = c.split()
        if words:
            nums = sum(1 for w in words if re.search(r'\d', w))
            number_raw[i] = nums / len(words)

    # 5. Discourse markers: count per token
    discourse_raw = np.zeros(n)
    for i, c in enumerate(chunks):
        lower_c = c.lower()
        words = c.split()
        count = sum(1 for m in DISCOURSE_MARKERS if m in lower_c)
        discourse_raw[i] = count / max(len(words), 1)

    # 6. Question presence
    question_raw = np.zeros(n)
    interrogatives = {'who', 'what', 'where', 'when', 'why', 'how'}
    for i, c in enumerate(chunks):
        has_qmark = '?' in c
        has_interrog = bool(set(c.lower().split()) & interrogatives)
        q_count = int(has_qmark) + int(has_interrog)
        question_raw[i] = min(1.0, q_count / 2.0)

    # Normalize each to [0, 1]
    def norm(arr):
        mx = arr.max()
        if mx > 0:
            return arr / mx
        return arr

    return {
        'entity_density':    norm(entity_raw),
        'tfidf_importance':  norm(tfidf_raw),
        'position_bias':     norm(position_raw),
        'numerical_density': norm(number_raw),
        'discourse_markers': norm(discourse_raw),
        'question_presence': norm(question_raw),
    }


def compute_salience(features_dict, weights=None):
    """
    Weighted sum of features. Pass a custom weights dict to ablate.
    Missing keys are treated as zero weight (feature removed).
    """
    if weights is None:
        weights = DEFAULT_WEIGHTS
    n = len(next(iter(features_dict.values())))
    scores = np.zeros(n)
    total_w = sum(weights.get(k, 0) for k in features_dict)
    if total_w == 0:
        return scores
    for feat_name, feat_arr in features_dict.items():
        w = weights.get(feat_name, 0)
        # Renormalize weights so they sum to ~0.9 (original total)
        scores += w * feat_arr
    return scores


def budgetmem_answer(document, question, budget_ratio=0.3, weights=None, top_k=3):
    """
    Full BudgetMem pipeline: chunk -> score -> select -> retrieve -> generate.
    """
    chunks = chunk_document(document)
    if not chunks:
        return generate_answer("", question)

    features = compute_features(chunks)
    salience = compute_salience(features, weights)

    k = max(1, int(len(chunks) * budget_ratio))
    top_indices = np.argsort(salience)[-k:][::-1]
    selected = [chunks[i] for i in sorted(top_indices)]

    retrieved = retrieve_bm25(question, selected, top_k=top_k)
    context = "\n\n".join(retrieved)
    return generate_answer(context, question)


print("BudgetMem scorer loaded (6 features, paper weights).")

---
## Part 1: Generate medium-length documents

Same structured papers used in the original experiments. We regenerate
them here so everything is self-contained.

In [ ]:
random.seed(42)
np.random.seed(42)

def create_research_paper(idx):
    """Generate a structured research paper with sections of varying salience."""

    topics = [
        'machine learning', 'deep learning', 'neural networks',
        'natural language processing', 'computer vision',
        'reinforcement learning', 'transfer learning', 'attention mechanisms',
        'graph neural networks', 'federated learning',
    ]
    methods = [
        'transformer', 'convolutional network', 'recurrent network',
        'generative adversarial network', 'variational autoencoder',
        'BERT', 'GPT', 'ResNet', 'diffusion model', 'mixture of experts',
    ]
    datasets_list = [
        'ImageNet', 'CIFAR-10', 'GLUE', 'SQuAD', 'COCO',
        'WMT', 'CommonCrawl', 'WikiText-103', 'Penn Treebank', 'LibriSpeech',
    ]

    topic  = topics[idx % len(topics)]
    method = methods[idx % len(methods)]
    ds     = datasets_list[idx % len(datasets_list)]
    acc    = round(85 + random.uniform(0, 12), 1)
    prev   = round(acc - random.uniform(2, 8), 1)
    lr     = round(random.choice([1e-3, 3e-4, 5e-4, 1e-4]), 5)
    bs     = random.choice([16, 32, 64, 128])
    epochs = random.choice([50, 100, 200, 300])

    abstract = (
        f"This paper presents a novel approach to {topic} using a {method} architecture. "
        f"We evaluate on {ds} and achieve {acc}% accuracy, surpassing the previous "
        f"state-of-the-art result of {prev}%. Our method requires no task-specific "
        f"fine-tuning and trains in under 24 hours on a single GPU. "
        f"We release code and pretrained weights to support reproducibility."
    )

    intro = (
        f"The field of {topic} has advanced rapidly over the past decade. "
        f"Classical approaches relied on hand-crafted feature pipelines that "
        f"demanded substantial domain expertise. The rise of deep learning "
        f"replaced many of these pipelines with end-to-end trainable models, "
        f"but challenges in scalability, generalization, and data efficiency "
        f"persist. Recent work on {method} architectures has shown promise, "
        f"yet existing implementations often require prohibitive compute budgets. "
        f"In this work we propose a lightweight variant that retains the "
        f"representational power of {method} while cutting training cost by 60%. "
        f"Our key contributions are: (1) a parameter-efficient adaptation scheme, "
        f"(2) a curriculum-based training strategy, and (3) extensive evaluation "
        f"on {ds} showing {acc}% accuracy. "
    ) * 8  # pad to ~1200 words

    related = (
        f"Prior work in {topic} spans several decades. Early statistical methods "
        f"achieved moderate success on constrained benchmarks but struggled with "
        f"real-world variability. The introduction of deep neural networks marked a "
        f"turning point, with convolutional and recurrent architectures dominating "
        f"leaderboards. More recently, attention-based models have set new records "
        f"across a range of tasks. However, these models are computationally "
        f"expensive and their environmental impact has drawn criticism. Several "
        f"groups have proposed efficiency improvements including pruning, "
        f"quantization, and knowledge distillation, but gains come at the cost of "
        f"accuracy. Our approach differs in that we modify the architecture itself "
        f"rather than applying post-hoc compression. "
    ) * 7

    methodology = (
        f"Our proposed {method} variant introduces three modifications to the "
        f"standard architecture. First, we replace dense layers with sparse "
        f"mixtures of experts, activating only 25% of parameters per forward pass. "
        f"Second, we apply rotary position embeddings to improve length "
        f"generalization. Third, we use a staged training curriculum that "
        f"begins with short sequences and gradually increases context length. "
        f"Training uses AdamW with a learning rate of {lr}, batch size {bs}, "
        f"for {epochs} epochs on 8 A100 GPUs. We apply gradient clipping at 1.0 "
        f"and use cosine learning rate decay with a 5% warmup period. "
        f"All hyperparameters were selected via grid search on a held-out "
        f"validation set of 5000 examples. "
    ) * 10

    results = (
        f"On {ds}, our model achieves {acc}% accuracy, compared to {prev}% "
        f"for the previous best method. The improvement is statistically "
        f"significant (p < 0.01, paired bootstrap test). Ablation studies confirm "
        f"that each of our three modifications contributes to the final result: "
        f"removing sparse experts drops accuracy to {round(acc-3.2,1)}%, removing "
        f"rotary embeddings to {round(acc-1.8,1)}%, and removing the curriculum to "
        f"{round(acc-2.5,1)}%. Training wall-clock time is 18 hours, versus 45 hours "
        f"for the dense baseline. Inference latency is 12ms per example on a "
        f"single V100 GPU. "
    ) * 7

    discussion = (
        f"Our results demonstrate that architectural efficiency and high accuracy "
        f"are not mutually exclusive in {topic}. The sparse mixture of experts "
        f"approach activates fewer parameters without sacrificing representational "
        f"capacity, and the training curriculum reduces wasted computation on "
        f"easy examples. However, limitations remain: our method has not been "
        f"tested on languages other than English, and the expert routing mechanism "
        f"adds implementation complexity. Future work will explore multilingual "
        f"settings and distillation of the sparse model into a smaller dense one. "
    ) * 5

    acknowledgements = (
        f"This work was supported by the National Science Foundation under grant "
        f"IIS-{random.randint(1800000,2100000)}. We thank the anonymous reviewers "
        f"for their constructive feedback and our colleagues for helpful discussions. "
        f"Computational resources were provided by the university HPC cluster."
    )

    paper = (
        f"Title: A Novel Approach to {topic.title()} Using {method.title()}\n\n"
        f"Abstract: {abstract}\n\n"
        f"1. Introduction\n{intro}\n\n"
        f"2. Related Work\n{related}\n\n"
        f"3. Methodology\n{methodology}\n\n"
        f"4. Results\n{results}\n\n"
        f"5. Discussion\n{discussion}\n\n"
        f"Acknowledgements\n{acknowledgements}"
    )

    # Questions targeting specific sections
    qa_pairs = [
        (f"What accuracy does the proposed method achieve on {ds}?", f"{acc}%"),
        (f"What was the previous state-of-the-art accuracy?", f"{prev}%"),
        (f"What learning rate was used for training?", f"{lr}"),
        (f"How many epochs was the model trained for?", f"{epochs}"),
        (f"What batch size was used during training?", f"{bs}"),
    ]
    return paper, qa_pairs


# Generate 200 QA pairs from 40 papers x 5 questions each
medium_docs = []
medium_qa = []
for idx in range(40):
    paper, qas = create_research_paper(idx)
    for q, a in qas:
        medium_docs.append(paper)
        medium_qa.append({'question': q, 'answer': a})

avg_tokens = np.mean([len(d.split()) for d in medium_docs[:40:5]])  # unique papers
print(f"Generated {len(medium_qa)} QA pairs from 40 papers.")
print(f"Average paper length: {avg_tokens:.0f} tokens")

---
## Part 2: Per-Feature Ablation (leave-one-out)

Run BudgetMem 7 times on the 200 medium-doc QA pairs:
- Once with all 6 features (full model)
- Once for each feature removed

This produces **Table 8** for the paper.

In [ ]:
def run_budgetmem_on_medium(docs, qa_pairs, weights, label=""):
    """Run BudgetMem with given weights on medium docs, return mean F1."""
    f1_scores = []
    for i in tqdm(range(len(qa_pairs)), desc=label):
        pred = budgetmem_answer(docs[i], qa_pairs[i]['question'], weights=weights)
        f1 = compute_f1(pred, qa_pairs[i]['answer'])
        f1_scores.append(f1)
    return np.mean(f1_scores), np.std(f1_scores), f1_scores


# Run full model first
print("Running full BudgetMem (all 6 features)...")
full_f1, full_std, full_scores = run_budgetmem_on_medium(
    medium_docs, medium_qa, DEFAULT_WEIGHTS, label="Full model"
)
print(f"Full model F1: {full_f1:.4f} (+/- {full_std:.4f})")

# Save checkpoint
with open(f"{PROJECT_DIR}/results/ablation_full.json", 'w') as f:
    json.dump({'f1_mean': full_f1, 'f1_std': full_std}, f)
print("Checkpoint saved.")

In [ ]:
# Leave-one-out ablation
feature_names = list(DEFAULT_WEIGHTS.keys())
ablation_results = {'full': {'f1_mean': full_f1, 'f1_std': full_std}}

for feat in feature_names:
    # Create weights with this feature zeroed out
    ablated_weights = dict(DEFAULT_WEIGHTS)
    ablated_weights[feat] = 0.0

    label = f"Without {feat}"
    print(f"\n{'='*60}")
    print(f"Ablation: {label}")
    print(f"{'='*60}")

    ab_f1, ab_std, _ = run_budgetmem_on_medium(
        medium_docs, medium_qa, ablated_weights, label=label
    )

    drop = full_f1 - ab_f1
    drop_pct = (drop / full_f1) * 100 if full_f1 > 0 else 0

    ablation_results[f'without_{feat}'] = {
        'f1_mean': ab_f1,
        'f1_std': ab_std,
        'f1_drop': drop,
        'f1_drop_pct': drop_pct,
    }
    print(f"{label}: F1 = {ab_f1:.4f} (drop = {drop:+.4f}, {drop_pct:+.1f}%)")

    # Save after each run in case Colab disconnects
    with open(f"{PROJECT_DIR}/results/ablation_results.json", 'w') as f:
        json.dump(ablation_results, f, indent=2)

print("\nAll ablation runs complete. Results saved.")

In [ ]:
# Print ablation table (this goes into the paper as Table 8)
print("\n" + "="*70)
print("TABLE: Per-Feature Ablation (Leave-One-Out)")
print("="*70)
print(f"{'Configuration':<35} {'F1 Score':<12} {'F1 Drop':<12} {'Drop %':<10}")
print("-"*70)

# Full model
print(f"{'Full BudgetMem (all features)':<35} {full_f1:<12.4f} {'--':<12} {'--':<10}")

# Each ablation, sorted by drop
ablations_sorted = []
for feat in feature_names:
    key = f'without_{feat}'
    r = ablation_results[key]
    ablations_sorted.append((feat, r['f1_mean'], r['f1_drop'], r['f1_drop_pct']))

ablations_sorted.sort(key=lambda x: x[2], reverse=True)  # biggest drop first

for feat, f1, drop, drop_pct in ablations_sorted:
    nice_name = feat.replace('_', ' ').title()
    label = f"  w/o {nice_name}"
    print(f"{label:<35} {f1:<12.4f} {drop:<12.4f} {drop_pct:<10.1f}%")

print("-"*70)
print("\nHighest-impact features (by F1 drop when removed):")
for feat, _, drop, drop_pct in ablations_sorted[:3]:
    print(f"  {feat}: -{drop_pct:.1f}%")

---
## Part 3: LLMLingua-2 on Medium Documents

This fills the gap in the paper: we never compared BudgetMem against
LLMLingua on the medium-length regime where BudgetMem shines.

In [ ]:
from llmlingua import PromptCompressor

print("Loading LLMLingua-2 compressor...")
llm_lingua = PromptCompressor(
    model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
    use_llmlingua2=True,
    device_map="cuda" if torch.cuda.is_available() else "cpu"
)
print("LLMLingua-2 loaded.")

In [ ]:
def run_llmlingua_on_medium(docs, qa_pairs, rate=0.3):
    """Run LLMLingua-2 compression + BM25 retrieval + Llama generation."""
    f1_scores = []
    errors = 0

    for i in tqdm(range(len(qa_pairs)), desc="LLMLingua-2 medium docs"):
        try:
            # Compress the document
            compressed = llm_lingua.compress_prompt(
                docs[i],
                rate=rate,
                force_tokens=['?', '.', '!', ','],
                drop_consecutive=True
            )
            compressed_text = compressed['compressed_prompt']

            # Chunk the compressed text, retrieve, generate
            chunks = chunk_document(compressed_text)
            if not chunks:
                chunks = [compressed_text]
            retrieved = retrieve_bm25(qa_pairs[i]['question'], chunks, top_k=3)
            context = "\n\n".join(retrieved)
            pred = generate_answer(context, qa_pairs[i]['question'])
            f1 = compute_f1(pred, qa_pairs[i]['answer'])
            f1_scores.append(f1)

        except Exception as e:
            errors += 1
            f1_scores.append(0.0)
            if errors <= 3:
                print(f"  Error on example {i}: {e}")

        # Save checkpoint every 50 examples
        if (i + 1) % 50 == 0:
            with open(f"{PROJECT_DIR}/results/llmlingua_medium_checkpoint.json", 'w') as f:
                json.dump({'completed': i+1, 'f1_so_far': float(np.mean(f1_scores))}, f)

    return np.mean(f1_scores), np.std(f1_scores), f1_scores, errors


# Also run baseline RAG (no compression) for a clean comparison
def run_baseline_on_medium(docs, qa_pairs):
    f1_scores = []
    for i in tqdm(range(len(qa_pairs)), desc="Baseline RAG medium docs"):
        chunks = chunk_document(docs[i])
        retrieved = retrieve_bm25(qa_pairs[i]['question'], chunks, top_k=3)
        context = "\n\n".join(retrieved)
        pred = generate_answer(context, qa_pairs[i]['question'])
        f1 = compute_f1(pred, qa_pairs[i]['answer'])
        f1_scores.append(f1)
    return np.mean(f1_scores), np.std(f1_scores), f1_scores

In [ ]:
# Run all three methods on medium docs
print("Running Baseline RAG on medium docs...")
base_f1, base_std, base_scores = run_baseline_on_medium(medium_docs, medium_qa)
print(f"Baseline RAG F1: {base_f1:.4f}")

print("\nRunning LLMLingua-2 on medium docs...")
ll_f1, ll_std, ll_scores, ll_errors = run_llmlingua_on_medium(medium_docs, medium_qa)
print(f"LLMLingua-2 F1: {ll_f1:.4f} (errors: {ll_errors})")

# BudgetMem already run in Part 2 (full_f1)
print(f"BudgetMem F1:   {full_f1:.4f} (from Part 2)")

In [ ]:
# Print comparison table for the paper
print("\n" + "="*70)
print("TABLE: Medium Documents - Three-Way Comparison (5K-10K tokens)")
print("="*70)
print(f"{'Method':<25} {'F1 Score':<12} {'Storage':<12} {'Neural?':<10}")
print("-"*60)
print(f"{'Baseline RAG':<25} {base_f1:<12.4f} {'100%':<12} {'--':<10}")
print(f"{'LLMLingua-2':<25} {ll_f1:<12.4f} {'30%':<12} {'Yes':<10}")
print(f"{'BudgetMem (Ours)':<25} {full_f1:<12.4f} {'30%':<12} {'No':<10}")
print("-"*60)

# Relative comparison
bm_vs_base = ((base_f1 - full_f1) / base_f1) * 100
ll_vs_base = ((base_f1 - ll_f1) / base_f1) * 100
bm_vs_ll = full_f1 - ll_f1

print(f"\nBudgetMem vs Baseline: {bm_vs_base:+.1f}% F1 drop")
print(f"LLMLingua vs Baseline: {ll_vs_base:+.1f}% F1 drop")
print(f"BudgetMem vs LLMLingua: {bm_vs_ll:+.4f} F1 difference")

# Save results
medium_comparison = {
    'experiment': 'Medium docs three-way comparison',
    'n_qa_pairs': len(medium_qa),
    'baseline_rag': {'f1_mean': float(base_f1), 'f1_std': float(base_std)},
    'llmlingua2':   {'f1_mean': float(ll_f1), 'f1_std': float(ll_std), 'errors': ll_errors},
    'budgetmem':    {'f1_mean': float(full_f1), 'f1_std': float(full_std)},
    'timestamp': datetime.now().isoformat()
}
with open(f"{PROJECT_DIR}/results/medium_comparison.json", 'w') as f:
    json.dump(medium_comparison, f, indent=2)
print("\nResults saved to medium_comparison.json")

---
## Part 4: Qasper Benchmark (Real Academic Papers)

This addresses the synthetic-data concern. Qasper contains
information-seeking questions on real NeurIPS papers (2-8K tokens).
We run Baseline RAG, BudgetMem, and LLMLingua-2 on 100 examples.

In [ ]:
# Downgrade datasets to support script-based datasets (Qasper)
!pip install -q "datasets>=2.20,<3"

# IMPORTANT: After this cell finishes, go to:
#   Runtime -> Restart session
# Then skip directly to the Qasper cell below (don't re-run this cell).
# Your earlier results are saved in /content/budgetmem_results/results/
print("Done. Now go to Runtime -> Restart session, then run the next cell.")

In [ ]:
# Re-import everything needed after runtime restart
import numpy as np
import json
import os
from datasets import load_dataset

PROJECT_DIR = "/content/budgetmem_results"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

# Reload earlier results if they exist
ablation_results = {}
medium_comparison = {}
if os.path.exists(f"{PROJECT_DIR}/results/ablation_results.json"):
    with open(f"{PROJECT_DIR}/results/ablation_results.json") as f:
        ablation_results = json.load(f)
    print(f"Loaded ablation results: {len(ablation_results)} entries")
if os.path.exists(f"{PROJECT_DIR}/results/medium_comparison.json"):
    with open(f"{PROJECT_DIR}/results/medium_comparison.json") as f:
        medium_comparison = json.load(f)
    print(f"Loaded medium comparison results")

print("Loading Qasper dataset...")
qasper = load_dataset("allenai/qasper", trust_remote_code=True)
print(f"Loaded Qasper. Validation set: {len(qasper['validation'])} papers.")

qasper_docs = []
qasper_qa = []

for paper in qasper['validation']:
    full_text = (paper.get('abstract', '') or '') + "\n\n"
    for sec_name, paragraphs in zip(
        paper['full_text']['section_name'],
        paper['full_text']['paragraphs']
    ):
        if sec_name:
            full_text += f"{sec_name}\n"
        full_text += "\n".join(paragraphs) + "\n\n"

    if len(full_text.split()) < 500:
        continue

    for q_idx in range(len(paper['qas']['question'])):
        question = paper['qas']['question'][q_idx]
        for ans_obj in paper['qas']['answers'][q_idx]['answer']:
            ans_text = ans_obj.get('free_form_answer', '')
            if not ans_text and ans_obj.get('extractive_spans'):
                ans_text = ' '.join(ans_obj['extractive_spans'])
            if ans_text and ans_text.strip().lower() not in ('', 'yes', 'no', 'unanswerable'):
                qasper_docs.append(full_text)
                qasper_qa.append({'question': question, 'answer': ans_text})
                break

    if len(qasper_qa) >= 100:
        break

print(f"Extracted {len(qasper_qa)} QA pairs from Qasper.")
avg_len = np.mean([len(d.split()) for d in qasper_docs])
print(f"Average paper length: {avg_len:.0f} tokens")

In [ ]:
# Run all three methods on Qasper
print("Running Baseline RAG on Qasper...")
qasper_base_f1, qasper_base_std, qasper_base_scores = run_baseline_on_medium(
    qasper_docs, qasper_qa
)
print(f"Baseline RAG F1: {qasper_base_f1:.4f}")

print("\nRunning BudgetMem on Qasper...")
qasper_bm_f1, qasper_bm_std, qasper_bm_scores = run_budgetmem_on_medium(
    qasper_docs, qasper_qa, DEFAULT_WEIGHTS, label="BudgetMem Qasper"
)
print(f"BudgetMem F1: {qasper_bm_f1:.4f}")

print("\nRunning LLMLingua-2 on Qasper...")
qasper_ll_f1, qasper_ll_std, qasper_ll_scores, qasper_ll_errs = run_llmlingua_on_medium(
    qasper_docs, qasper_qa
)
print(f"LLMLingua-2 F1: {qasper_ll_f1:.4f}")

In [ ]:
# Print Qasper results
print("\n" + "="*70)
print("TABLE: Qasper Benchmark (Real Academic Papers)")
print("="*70)
print(f"{'Method':<25} {'F1 Score':<12} {'Storage':<12} {'Neural?':<10}")
print("-"*60)
print(f"{'Baseline RAG':<25} {qasper_base_f1:<12.4f} {'100%':<12} {'--':<10}")
print(f"{'LLMLingua-2':<25} {qasper_ll_f1:<12.4f} {'30%':<12} {'Yes':<10}")
print(f"{'BudgetMem (Ours)':<25} {qasper_bm_f1:<12.4f} {'30%':<12} {'No':<10}")
print("-"*60)

bm_drop = ((qasper_base_f1 - qasper_bm_f1) / qasper_base_f1) * 100 if qasper_base_f1 > 0 else 0
print(f"\nBudgetMem vs Baseline: {bm_drop:+.1f}% F1 drop")
print(f"Storage savings: ~70%")

# Save results
qasper_results = {
    'experiment': 'Qasper benchmark (real academic papers)',
    'n_qa_pairs': len(qasper_qa),
    'avg_doc_length': float(avg_len),
    'baseline_rag': {'f1_mean': float(qasper_base_f1), 'f1_std': float(qasper_base_std)},
    'llmlingua2':   {'f1_mean': float(qasper_ll_f1), 'f1_std': float(qasper_ll_std)},
    'budgetmem':    {'f1_mean': float(qasper_bm_f1), 'f1_std': float(qasper_bm_std)},
    'timestamp': datetime.now().isoformat()
}
with open(f"{PROJECT_DIR}/results/qasper_results.json", 'w') as f:
    json.dump(qasper_results, f, indent=2)
print("\nResults saved to qasper_results.json")

---
## Part 5: Full Summary

In [ ]:
print("\n" + "="*70)
print("REVISION EXPERIMENTS - COMPLETE SUMMARY")
print("="*70)

print("\n--- 1. Per-Feature Ablation (medium docs, 200 QA pairs) ---")
print(f"Full BudgetMem F1: {full_f1:.4f}")
for feat, f1, drop, drop_pct in ablations_sorted:
    nice = feat.replace('_', ' ').title()
    print(f"  w/o {nice:<25} F1={f1:.4f}  drop={drop_pct:+.1f}%")

print("\n--- 2. Medium Docs Three-Way Comparison ---")
print(f"Baseline RAG:   {base_f1:.4f}")
print(f"LLMLingua-2:    {ll_f1:.4f}")
print(f"BudgetMem:      {full_f1:.4f}")

print("\n--- 3. Qasper Benchmark (real papers) ---")
print(f"Baseline RAG:   {qasper_base_f1:.4f}")
print(f"LLMLingua-2:    {qasper_ll_f1:.4f}")
print(f"BudgetMem:      {qasper_bm_f1:.4f}")

# Save everything in one file
all_revision_results = {
    'ablation': ablation_results,
    'medium_comparison': medium_comparison,
    'qasper': qasper_results,
    'timestamp': datetime.now().isoformat()
}
with open(f"{PROJECT_DIR}/results/ALL_REVISION_RESULTS.json", 'w') as f:
    json.dump(all_revision_results, f, indent=2)

print(f"\nAll results saved to {PROJECT_DIR}/results/ALL_REVISION_RESULTS.json")
print("\nShare this file back and I will update the paper with the new tables.")